# Gold scoring export

Writes `gold_renewal_scores.parquet` / `.csv` for Power BI Import mode (`../data/gold` relative to repo root).

`score_run()` reads training metadata (`artifacts/last_train.json`) to decide between MLflow and local pickle.

Job parameters (optional): `EXPORT_CSV_FOR_PBIP`, `gold_csv_workspace_export` — when the export path is set (e.g. `/Workspace/Users/you@corp/gold_renewal_scores.csv`), a **copy** of the CSV is written there for BI pick-up.

In [ ]:
%pip install -q pandas pyarrow joblib mlflow numpy scikit-learn
import os, sys, shutil
from pathlib import Path

REPO = os.environ.get("PORTFOLIO_REPO_ROOT", "../..")
sys.path.insert(0, f"{REPO}/python")

if "dbutils" in globals():
    dbutils.widgets.text("base_path", "")  # type: ignore
    dbutils.widgets.dropdown("EXPORT_CSV_FOR_PBIP", "true", ["true", "false"])  # type: ignore
    dbutils.widgets.text("gold_csv_workspace_export", "")  # type: ignore

if os.environ.get("PORTFOLIO_BASE_PATH") is None and "dbutils" in globals():
    _bp = dbutils.widgets.get("base_path")  # type: ignore
    if _bp:
        os.environ["PORTFOLIO_BASE_PATH"] = _bp

from cs_portfolio.score import score_run

csv_path = Path(score_run())
print({"gold_csv": str(csv_path)})

export_to = ""
if "dbutils" in globals():
    export_to = dbutils.widgets.get("gold_csv_workspace_export").strip()  # type: ignore
if export_to:
    dest = Path(export_to)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(csv_path, dest)
    print({"copied_gold_csv_to_workspace": str(dest)})

### Optional Delta hand-off

Expose the scored table to `GoldRenewalScores` SQL endpoint / DirectLake by writing `scores` Delta + UC GRANT SELECT to the BI security group documented in governance.